<a href="https://colab.research.google.com/github/mhw47/ML_Practice/blob/main/HousePricePredictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
import pandas as pd
import os
import sklearn
from data_profiling import ProfileReport

In [18]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("tanyachawla412/house-prices")

print("Path to dataset files:", path)
csv_path = "/kaggle/input/house-prices/House_price.csv"

Using Colab cache for faster access to the 'house-prices' dataset.
Path to dataset files: /kaggle/input/house-prices


In [19]:
from google.colab import userdata

# Pass your Kaggle credentials to the environment
os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')


In [20]:
df = pd.read_csv(csv_path)

In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4548 entries, 0 to 4547
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Avg. Area Income    4548 non-null   float64
 1   House Age           4548 non-null   float64
 2   Number of Rooms     4548 non-null   float64
 3   Number of Bedrooms  4548 non-null   float64
 4   Area Population     4548 non-null   float64
 5   Price               4548 non-null   float64
 6   Address             4548 non-null   object 
dtypes: float64(6), object(1)
memory usage: 248.8+ KB


In [22]:
profile = ProfileReport(df, title="Exploratory Data Analysis Report")
profile.to_file("data_profile_report.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 7/7 [00:00<00:00, 25.35it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [23]:
df.shape

(4548, 7)

In [24]:
df.sample(5)

,Avg. Area Income,House Age,Number of Rooms,Number of Bedrooms,Area Population,Price,Address
2233,74614.82449,5.008817,5.963195,4.31,46070.44340,1.033881e+06,"933 David Station Suite 518\nTiffanyborough, N..."
1110,80284.99543,5.029475,4.049321,2.20,23457.12640,6.379519e+05,"9488 Wood Hill\nNew Jesseton, IL 19753"
1659,55543.52009,6.793799,7.872303,5.41,40063.42110,1.193483e+06,"9361 Stephen Stream\nPort Meganfurt, AR 82577"
199,59362.76638,7.482630,8.003007,3.49,44723.35373,1.442945e+06,"8581 Johnson Ranch Apt. 905\nJacquelinefurt, M..."
792,64290.20571,3.648546,7.368073,3.44,54157.29516,9.490722e+05,"519 Avila Road\nJamesbury, SC 76217-0455"


In [25]:
from sklearn.model_selection import train_test_split

In [26]:
# 1. Separate features (X) and target (Y)
# We drop 'Price' and 'Address' as 'Address' is non-numeric text
X = df.drop(columns=['Price', 'Address'])
Y = df['Price']

In [27]:
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, r2_score

# 1. Split plain X and Y (90% study, 10% test)
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.1, random_state=42)

# 2. Pick the brain
xgb_model = xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42)

# 3. Study (.fit)
xgb_model.fit(X_train, Y_train)

# 4. Take the test (.predict)
Y_pred = xgb_model.predict(X_test)

# 5. Grade the test (Directly in real dollars—no multiplying by 1,000,000 needed!)
r2 = r2_score(Y_test, Y_pred)
mae = mean_absolute_error(Y_test, Y_pred)

print(f"R2 Score: {r2:.2%}")
print(f"Average Dollar Error (MAE): ${mae:,.2f}")

R2 Score: 89.01%
Average Dollar Error (MAE): $91,842.84


In [28]:
custom_sample = pd.DataFrame([{
    'Avg. Area Income': 70000.0,
    'House Age': 6.0,
    'Number of Rooms': 7.0,
    'Number of Bedrooms': 4.0,
    'Area Population': 35000.0
}])

# Predict directly on the custom house without scaling!
predicted_price = xgb_model.predict(custom_sample)[0]
print(f"Predicted House Price: ${predicted_price:,.2f}")

Predicted House Price: $1,253,358.00
